# ES4304 Tutorial 1: Sentinel-2 Data Visualisation
## Mount Merapi Eruption — 29 March 2023

This notebook processes Sentinel-2 L1C imagery of the Mount Merapi eruption (11 March 2023) and creates georeferenced GeoTIFF files for analysis in QGIS.

**Scene:** Sentinel-2A, 29 March 2023, Tile T49MDM (18 days post-eruption)  
**Processing Baseline:** N0509 — a radiometric offset of +1000 DN is applied to all bands. The correction formula is: `Reflectance = (DN − 1000) / 10000`

### Outputs produced by this notebook

| Output file | Bands | Resolution | Purpose |
|---|---|---|---|
| `B02_B03_B04_B08_10m.tif` | Blue, Green, Red, NIR | 10 m | True colour + vegetation |
| `B05_B06_B07_B8A_B11_B12_20m.tif` | Red Edge, NIR narrow, SWIR | 20 m | Spectral analysis |
| `B01_B09_B10_60m.tif` | Aerosol, Water Vapour, Cirrus | 60 m | Atmospheric reference |
| `NDVI_10m.tif` | Derived: (B08 − B04) / (B08 + B04) | 10 m | Vegetation health |
| `SWIR-RGB_20m.tif` | B12 → R, B11 → G, B04 → B | 20 m | Pyroclastic flow detection |
| `NBR_20m.tif` | Derived: (B8A − B12) / (B8A + B12) | 20 m | Burn severity |

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import resize
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_origin
import urllib.request
import zipfile
import xml.etree.ElementTree as ET

## Helper Functions

Python equivalents of the original MATLAB scripts:
- `create_tif_variable_From_JP2_function.m`
- `save_As_GeoTiff_function.m`

**Note on `flipud`:** The original MATLAB code applies `flipud()` because MATLAB's `imread` reads Sentinel-2 JP2 files south-up (row 0 = south). Rasterio reads JP2 files **north-up natively** (row 0 = north), so no flipping is needed here.

In [ ]:
def create_tif_variable_from_jp2(img_data_dir, *filenames):
    """
    Read one or more Sentinel-2 JP2 band files from img_data_dir and
    stack them into a single (rows, cols, num_bands) array, north-up.

    Bands are kept at their native uint16 dtype and decoded directly
    into the output array — no per-band temporary and no np.stack copy
    at the end. On a full 10980x10980 tile that is the difference
    between a ~3.9 GB peak and a ~1.0 GB one.
    """
    paths = [Path(img_data_dir) / filename for filename in filenames]

    with rasterio.open(paths[0]) as src:
        rows, cols = src.height, src.width

    # Shaped (bands, rows, cols) so each band decodes straight into its
    # own contiguous slot. The transpose on return is a view, not a copy.
    img_data = np.empty((len(paths), rows, cols), dtype=np.uint16)

    # JPEG2000 decoding is single-threaded by default; ALL_CPUS lets the
    # OpenJPEG driver decode codestream tiles across every core.
    with rasterio.Env(GDAL_NUM_THREADS='ALL_CPUS'):
        for band_idx, path in enumerate(paths):
            with rasterio.open(path) as src:
                src.read(1, out=img_data[band_idx])

    return img_data.transpose(1, 2, 0)

In [ ]:
def save_as_geotiff(composite_image, ulx, uly, xdim, ydim, crs_code, output_filename):
    """
    Save a composite image array as a georeferenced GeoTIFF file.

    Output is tiled and DEFLATE-compressed: a full-tile 4-band uint16
    stack drops from ~1 GB to roughly half that on disk, and tiled
    blocks let QGIS pan without re-reading whole scanlines.
    """
    if composite_image.ndim == 2:
        composite_image = composite_image[:, :, np.newaxis]

    rows, cols, num_bands = composite_image.shape
    transform = from_origin(ulx, uly, xdim, abs(ydim))
    crs = CRS.from_epsg(crs_code)

    # Horizontal differencing predictor: 2 for integers, 3 for floats.
    predictor = 3 if np.issubdtype(composite_image.dtype, np.floating) else 2

    with rasterio.open(
        output_filename, 'w',
        driver='GTiff',
        height=rows, width=cols,
        count=num_bands,
        dtype=composite_image.dtype,
        crs=crs,
        transform=transform,
        tiled=True, blockxsize=512, blockysize=512,
        compress='DEFLATE', predictor=predictor,
        num_threads='ALL_CPUS',
    ) as dst:
        for band_idx in range(num_bands):
            dst.write(
                np.ascontiguousarray(composite_image[:, :, band_idx]),
                band_idx + 1,
            )

    print(f'Saved: {output_filename}')

In [ ]:
def find_img_data(safe_dir):
    matches = list(Path(safe_dir).glob('GRANULE/*/IMG_DATA'))
    if not matches:
        raise FileNotFoundError(f'No IMG_DATA folder found in {safe_dir}')
    return matches[0]

def find_tile_date_prefix(img_data_dir):
    matches = list(Path(img_data_dir).glob('*_B02.jp2'))
    if not matches:
        raise FileNotFoundError(f'No *_B02.jp2 found in {img_data_dir}')
    return matches[0].stem.replace('_B02', '')

def find_tci(img_data_dir):
    matches = list(Path(img_data_dir).glob('*_TCI.jp2'))
    if not matches:
        raise FileNotFoundError(f'No *_TCI.jp2 found in {img_data_dir}')
    return matches[0].name

def read_metadata(safe_dir):
    matches = list(Path(safe_dir).glob('GRANULE/*/MTD_TL.xml'))
    if not matches:
        raise FileNotFoundError(f'No MTD_TL.xml found in {safe_dir}')
    root = ET.parse(matches[0]).getroot()
    ulx  = float(root.findall('.//{*}ULX')[0].text)
    uly  = float(root.findall('.//{*}ULY')[0].text)
    epsg = int(root.findall('.//{*}HORIZONTAL_CS_CODE')[0].text.split(':')[-1])
    return ulx, uly, epsg

### Radiometric Correction: DN to Reflectance

Sentinel-2 scenes processed with **Processing Baseline N0400 and later** (including the N0509 scene used here) store Digital Numbers with a **+1000 offset** to accommodate slightly negative TOA reflectance values.

```
Reflectance = (DN − 1000) / 10000
```

Values are clipped to [0, 1] for display. Note: TOA reflectance can legitimately exceed 1.0 over bright cloud tops — the clip is applied only for visualisation purposes.

In [ ]:
def dn_to_reflectance(x, offset=1000, ratio=10000.0):
    """
    Convert Sentinel-2 DN values to surface reflectance.

    Applies the radiometric offset introduced in Processing Baseline N0400:
        Reflectance = (DN - offset) / ratio

    Values are clipped to [0, 1] for display purposes.

    Parameters
    ----------
    x : array-like
        Raw DN values (int16, int32, or float).
    offset : int
        Radiometric offset — 1000 for baselines N0400 and later.
    ratio : float
        Scale factor — 10000 for Sentinel-2 L1C.

    Returns
    -------
    numpy.ndarray of float32, clipped to [0, 1].
    """
    return np.clip((np.float32(x) - offset) / ratio, 0, 1)

## Downloading the Scene (.SAFE)

If you haven't downloaded and unzipped the scene yet, run the cell below. It works the same way on **Mac, Linux, and Windows** — it uses Python's built-in `urllib` instead of `wget`, since `wget` isn't installed by default on Windows.

The cell will:
1. Download `S2A_MSIL1C_20230329T023531_N0509_R089_T49MDM_20230329T045534.SAFE.zip` into `data/` (skipped if already downloaded)
2. Unzip it into `data/...SAFE/` (skipped if already extracted)

> If you prefer the command-line approach: on Mac/Linux you can instead run `wget https://eyes-on-earth-tutorial-resources.s3.ap-southeast-1.amazonaws.com/tutorial_1/1.1_Merapi_Eruption_Data/S2A_MSIL1C_20230329T023531_N0509_R089_T49MDM_20230329T045534.SAFE.zip` in a terminal, then unzip the file into `data/`.

In [ ]:
scene_url = (
    "https://eyes-on-earth-tutorial-resources.s3.ap-southeast-1.amazonaws.com/"
    "tutorial_1/1.1_Merapi_Eruption_Data/"
    "S2A_MSIL1C_20230329T023531_N0509_R089_T49MDM_20230329T045534.SAFE.zip"
)

# Root directory containing the unzipped .SAFE folder (same path used in
# the Configuration cell below — defined here too so this cell can be
# run on its own, before Configuration has run).
directory = Path.cwd() / "data/"
directory.mkdir(parents=True, exist_ok=True)

zip_path = directory / Path(scene_url).name
safe_dir = directory / zip_path.stem  # strips the .zip extension

def _report_progress(block_num, block_size, total_size):
    downloaded = block_num * block_size
    if total_size > 0:
        pct = min(downloaded / total_size * 100, 100)
        print(f"\rDownloading... {pct:5.1f}% ({downloaded / 1e6:,.0f} / {total_size / 1e6:,.0f} MB)", end="")

if safe_dir.exists():
    print(f"Already extracted: {safe_dir}")
elif zip_path.exists():
    print(f"Zip already downloaded: {zip_path}\nExtracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(directory)
    print(f"Extracted to: {safe_dir}")
else:
    print(f"Downloading from:\n{scene_url}\n")
    urllib.request.urlretrieve(scene_url, zip_path, reporthook=_report_progress)
    print(f"\nSaved: {zip_path}\n\nExtracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(directory)
    print(f"Extracted to: {safe_dir}")

## Configuration

Update only the `.SAFE` folder name below — everything else
(IMG_DATA path, tile prefix, ULX, ULY, CRS) is auto-detected.

In [ ]:
# Only change this to match your downloaded .SAFE folder name
safe_folder = "S2A_MSIL1C_20230329T023531_N0509_R089_T49MDM_20230329T045534.SAFE"
data_dir    = Path.cwd() / "data"

# Everything below is auto-detected
safe_dir             = data_dir / safe_folder
img_data_dir         = find_img_data(safe_dir)
prefix               = find_tile_date_prefix(img_data_dir)
ULX, ULY, crsCode   = read_metadata(safe_dir)

print(f"IMG_DATA : {img_data_dir.relative_to(data_dir)}")
print(f"Prefix   : {prefix}")
print(f"ULX/ULY  : {ULX}, {ULY}  (EPSG:{crsCode})")

Path("outputs").mkdir(exist_ok=True)
print(f"Output   : {Path('outputs').resolve()}")

## True Colour Image (TCI)

The TCI is a pre-processed RGB composite (B04, B03, B02) provided directly by ESA alongside the raw bands. It gives a natural-colour view similar to what the human eye would see.

**Tutorial Step 3:** Load the original `TCI.jp2` file directly into QGIS from inside the IMG_DATA folder — no processing is needed. The cell below reads it into memory for the visualisation preview at the end of this notebook.

In [ ]:
tci_filename = find_tci(img_data_dir)
print(f"TCI file : {tci_filename}")

with rasterio.open(img_data_dir / tci_filename) as src:
    tci_10m = src.read().transpose(1, 2, 0)

print(f"tci_10m shape: {tci_10m.shape}  dtype: {tci_10m.dtype}")

## 10 m Resolution Bands: B02, B03, B04, B08

| Band | Name | Wavelength | Primary use in this tutorial |
|---|---|---|---|
| B02 | Blue | 490 nm | True colour display |
| B03 | Green | 560 nm | True colour display |
| B04 | Red | 665 nm | NDVI (Red channel); SWIR-RGB (Blue channel) |
| B08 | NIR | 842 nm | NDVI (NIR channel) |

Running this cell saves `B02_B03_B04_B08_10m.tif`, which can be loaded into QGIS as a false-colour composite if desired.

In [ ]:
bands_10m = [
    f'{prefix}_B02.jp2',   # Band 2  — Blue   (490 nm, 10 m)
    f'{prefix}_B03.jp2',   # Band 3  — Green  (560 nm, 10 m)
    f'{prefix}_B04.jp2',   # Band 4  — Red    (665 nm, 10 m)
    f'{prefix}_B08.jp2',   # Band 8  — NIR    (842 nm, 10 m)
]

tif_10m = create_tif_variable_from_jp2(img_data_dir, *bands_10m)
print(f"tif_10m shape: {tif_10m.shape}  dtype: {tif_10m.dtype}")
save_as_geotiff(tif_10m, ULX, ULY, 10, -10, crsCode, 'outputs/B02_B03_B04_B08_10m.tif')

## 20 m Resolution Bands: B05, B06, B07, B8A, B11, B12

| Band | Name | Wavelength | Primary use in this tutorial |
|---|---|---|---|
| B05 | Red Edge 1 | 705 nm | Vegetation stress |
| B06 | Red Edge 2 | 740 nm | Vegetation stress |
| B07 | Red Edge 3 | 783 nm | Vegetation stress |
| B8A | NIR narrow | 865 nm | Loaded for spectral completeness |
| B11 | SWIR-1 | 1610 nm | SWIR-RGB (Green channel); soil moisture |
| B12 | SWIR-2 | 2190 nm | SWIR-RGB (Red channel); ash, burn scars |

**B11 and B12 are the key bands for detecting pyroclastic flow deposits** — fresh volcanic material has a distinctive SWIR signature.

In [ ]:
bands_20m = [
    f'{prefix}_B05.jp2',   # Band 5  — Red Edge 1 (705 nm, 20 m)
    f'{prefix}_B06.jp2',   # Band 6  — Red Edge 2 (740 nm, 20 m)
    f'{prefix}_B07.jp2',   # Band 7  — Red Edge 3 (783 nm, 20 m)
    f'{prefix}_B8A.jp2',   # Band 8A — NIR narrow (865 nm, 20 m)
    f'{prefix}_B11.jp2',   # Band 11 — SWIR-1    (1610 nm, 20 m)
    f'{prefix}_B12.jp2',   # Band 12 — SWIR-2    (2190 nm, 20 m)
]

tif_20m = create_tif_variable_from_jp2(img_data_dir, *bands_20m)
print(f"tif_20m shape: {tif_20m.shape}  dtype: {tif_20m.dtype}")
save_as_geotiff(tif_20m, ULX, ULY, 20, -20, crsCode, 'outputs/B05_B06_B07_B8A_B11_B12_20m.tif')

## 60 m Resolution Bands: B01, B09, B10

These are atmospheric correction bands used by ESA's processing chain. They are included here to match the complete output of the original MATLAB script.

| Band | Name | Wavelength | Purpose |
|---|---|---|---|
| B01 | Coastal Aerosol | 443 nm | Aerosol and haze estimation |
| B09 | Water Vapour | 945 nm | Atmospheric water vapour |
| B10 | Cirrus | 1375 nm | Thin cirrus cloud detection |

> **Note:** These bands are not used directly in the tutorial analysis (NDVI or SWIR-RGB). They are provided for reference only.

In [ ]:
bands_60m = [
    f'{prefix}_B01.jp2',   # Band 1  — Coastal Aerosol (443 nm, 60 m)
    f'{prefix}_B09.jp2',   # Band 9  — Water Vapour   (945 nm, 60 m)
    f'{prefix}_B10.jp2',   # Band 10 — Cirrus         (1375 nm, 60 m)
]

tif_60m = create_tif_variable_from_jp2(img_data_dir, *bands_60m)
print(f"tif_60m shape: {tif_60m.shape}  dtype: {tif_60m.dtype}")
save_as_geotiff(tif_60m, ULX, ULY, 60, -60, crsCode, 'outputs/B01_B09_B10_60m.tif')

## SWIR-RGB Composite

The SWIR composite assigns shortwave infrared bands to the RGB display channels, making volcanic and fire-affected surfaces clearly distinguishable from vegetation.

| Display channel | Band | Wavelength | What it highlights |
|---|---|---|---|
| **Red** | B12 (SWIR-2) | 2190 nm | Volcanic ash, bare rock, burned surfaces |
| **Green** | B11 (SWIR-1) | 1610 nm | Soil moisture, disturbed land |
| **Blue** | B04 (Red) | 665 nm | Vegetation chlorophyll |

This matches the **Copernicus browser SWIR composite** shown in Tutorial Section 3, Step 5. In this composite, healthy vegetation appears **green**, pyroclastic deposits appear **red-orange**, and clouds appear **white**. This is the more reliable tool for identifying pyroclastic flow deposits compared to NDVI, because fresh volcanic material has a distinct SWIR spectral signature that stands out even through thin cloud and haze.

**Resolution mismatch:** B04 is 10 m; B11 and B12 are 20 m. B04 is downsampled to 20 m using **bilinear interpolation** with anti-aliasing (`skimage.transform.resize`, `order=1`) before compositing.

**Tutorial Step 8:** Load `SWIR-RGB_20m.tif` into QGIS with a basemap and compare against what you observed in the Copernicus browser.

In [ ]:
# B04 is 10m — downsample to 20m to match B11 and B12 resolution.
# resize() preserves DN value range; dn_to_reflectance() is applied afterwards.
b04 = tif_10m[:, :, 2].astype(np.float32)
b04_20m = resize(b04, tif_20m.shape[:2], order=1, anti_aliasing=True)

# Band indices in tif_20m (0-based):
#   index 4 → B11 (SWIR-1, 1610 nm)   ← Green channel
#   index 5 → B12 (SWIR-2, 2190 nm)   ← Red channel
swir_rgb_20m = np.dstack([
    dn_to_reflectance(tif_20m[:, :, 5]),   # B12 (SWIR-2) → R
    dn_to_reflectance(tif_20m[:, :, 4]),   # B11 (SWIR-1) → G
    dn_to_reflectance(b04_20m),            # B04 (Red)    → B
])
print(f"swir_rgb_20m shape: {swir_rgb_20m.shape}  dtype: {swir_rgb_20m.dtype}")
save_as_geotiff(swir_rgb_20m, ULX, ULY, 20, -20, crsCode, 'outputs/SWIR-RGB_20m.tif')

## NDVI — Normalized Difference Vegetation Index

NDVI measures vegetation health using the contrast between NIR (strongly reflected by healthy vegetation) and Red (absorbed by chlorophyll).

$$NDVI = \frac{NIR - Red}{NIR + Red} = \frac{B08 - B04}{B08 + B04}$$

| NDVI value | Surface type |
|---|---|
| 0.6 – 0.9 | Dense tropical forest |
| 0.3 – 0.6 | Sparse vegetation / crops |
| 0.0 – 0.3 | Bare soil, rock |
| < 0.0 | Water, cloud shadows |

**Band indices in `tif_10m` (Python 0-based):**
- `tif_10m[:, :, 3]` = B08 (NIR) — equivalent to MATLAB's `tif_10m(:,:,4)`
- `tif_10m[:, :, 2]` = B04 (Red) — equivalent to MATLAB's `tif_10m(:,:,3)`

**Limitation for this scene:** Java in late March is heavily clouded. Cloud tops produce ambiguous NDVI values that cannot be distinguished from bare volcanic deposits — NDVI alone might not reliably identify pyroclastic flows. Use the SWIR-RGB composite for volcanic signal detection.

**Tutorial Step 8:** Load `NDVI_10m.tif` into QGIS. Apply a **RdYlGn** diverging colour ramp with Min = −1, Max = +1.

In [ ]:
# Apply DN offset correction before computing NDVI
# N0509 baseline: (DN - 1000) / 10000 gives true TOA reflectance
nir = (tif_10m[:, :, 3].astype(np.float32) - 1000) / 10000.0  # B08
red = (tif_10m[:, :, 2].astype(np.float32) - 1000) / 10000.0  # B04

with np.errstate(invalid='ignore'):   # suppress expected 0/0 warning
    denominator = nir + red
    ndvi = np.where(
        denominator != 0,
        (nir - red) / denominator,
        0.0,                          # no-data pixels → 0.0 (not NaN)
    )
print(f"ndvi shape: {ndvi.shape}  min: {ndvi.min():.3f}  max: {ndvi.max():.3f}")
save_as_geotiff(ndvi, ULX, ULY, 10, -10, crsCode, 'outputs/NDVI_10m.tif')

## NBR — Normalized Burn Ratio

NBR detects burned areas using the contrast between NIR (high in healthy vegetation) and SWIR (high in burned, bare, or mineralogically altered surfaces).

$$NBR = \frac{NIR - SWIR}{NIR + SWIR} = \frac{B8A - B12}{B8A + B12}$$

| NBR value | Surface type |
|---|---|
| 0.6 – 0.9 | Healthy vegetation |
| 0.1 – 0.6 | Sparse vegetation / bare soil |
| −0.1 – 0.1 | Bare rock, ash deposits |
| < −0.1 | Burned areas, active fire scars |

**Why B8A and not B8?**  
B8A (865 nm, 20 m) is used instead of B8 (842 nm, 10 m) because it matches the spatial resolution of B12 (20 m). Mixing 10 m and 20 m bands would require resampling.

**Band indices in `tif_20m` (Python 0-based):**
- `tif_20m[:, :, 3]` = B8A (NIR narrow) — equivalent to MATLAB's `tif_20m(:,:,4)`
- `tif_20m[:, :, 5]` = B12 (SWIR-2) — equivalent to MATLAB's `tif_20m(:,:,6)`

**Tutorial Step 8:** Load `NBR_20m.tif` into QGIS alongside the NDVI layer. Apply a **RdYlGn** diverging colour ramp with Min = −1, Max = +1.

In [ ]:
# ── NBR ─────────────────────────────────────────────────────
# Formula : (B8A - B12) / (B8A + B12)
# Indices : tif_20m[:,:,3] = B8A (NIR narrow), tif_20m[:,:,5] = B12 (SWIR-2)
# B8A (20 m) is used instead of B8 (10 m) to match B12 resolution.
print("  Computing NBR...")
nir_b8a  = (tif_20m[:, :, 3].astype(np.float32) - 1000) / 10000.0
swir_b12 = (tif_20m[:, :, 5].astype(np.float32) - 1000) / 10000.0
with np.errstate(invalid='ignore'):
    denom_nbr = nir_b8a + swir_b12
    nbr_20m = np.where(denom_nbr != 0, (nir_b8a - swir_b12) / denom_nbr, 0.0)
    
print(f"nbr shape: {nbr_20m.shape}  min: {nbr_20m.min():.3f}  max: {nbr_20m.max():.3f}")
save_as_geotiff(nbr_20m, ULX, ULY, 20, -20, crsCode, 'outputs/NBR_20m.tif')

## Visualisation

Side-by-side preview of the three key outputs, cropped to the Merapi summit area.

> The TCI and NDVI crops use 10 m pixel indices `[3000:3800, 3400:4200]` (800 × 800 px = 8 km × 8 km).  
> The SWIR-RGB crop uses 20 m pixel indices `[1500:1900, 1700:2100]` (400 × 400 px = 8 km × 8 km).  
> All windows cover the same geographic area.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].set_title("TCI (True Colour)", fontsize=12)
axes[0, 0].imshow(tci_10m[3000:3800, 3400:4200])
axes[0, 0].set_xlabel("10 m pixels")

axes[0, 1].set_title("SWIR-RGB (B12, B11, B04)", fontsize=12)
axes[0, 1].imshow(swir_rgb_20m[1500:1900, 1700:2100, :])
axes[0, 1].set_xlabel("20 m pixels")

axes[1, 0].set_title("NDVI", fontsize=12)
im_ndvi = axes[1, 0].imshow(ndvi[3000:3800, 3400:4200], cmap="RdYlGn", vmin=-1, vmax=1)
axes[1, 0].set_xlabel("10 m pixels")
plt.colorbar(im_ndvi, ax=axes[1, 0], label='NDVI')

axes[1, 1].set_title("NBR", fontsize=12)
im_nbr = axes[1, 1].imshow(nbr_20m[1500:1900, 1700:2100], cmap="RdYlGn", vmin=-1, vmax=1)
axes[1, 1].set_xlabel("20 m pixels")
plt.colorbar(im_nbr, ax=axes[1, 1], label='NBR')

plt.tight_layout()
plt.show()